[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/10_moe_and_routing.ipynb)

# 10. Mixture-of-Experts and routing — from dense FFN to paper-faithful DeepSeekMoE

이 노트북은 router가 expert id만 고르는 toy를 넘어서 실제 token dispatch와 weighted combine을 구현한다.

이번 버전에서는 DeepSeekMoE의 두 핵심 구조를 작은 dimension으로 유지한다.

1. **fine-grained expert segmentation**: 기존 `N`개 expert를 `mN`개의 더 작은 expert로 쪼개고 `mK`개를 활성화한다.
2. **shared expert isolation**: 일부 expert는 모든 token에 항상 적용하고 나머지만 routing한다.

expert 수와 hidden size는 작게 줄이지만 계산 구조는 논문과 동일하게 유지한다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Dense SwiGLU FFN baseline

DeepSeek 계열 expert는 일반적인 `Linear -> activation -> Linear`보다 SwiGLU 형태에 가깝다.

`FFN(x) = W2( SiLU(W1 x) * W3 x )`


In [ ]:
class SwiGLUFFN(nn.Module):
    def __init__(
        self,
        model_dim,
        hidden_dim,
    ):
        super().__init__()

        self.gate_projection = nn.Linear(
            model_dim,
            hidden_dim,
            bias=False,
        )
        self.value_projection = nn.Linear(
            model_dim,
            hidden_dim,
            bias=False,
        )
        self.output_projection = nn.Linear(
            hidden_dim,
            model_dim,
            bias=False,
        )

    def forward(self, x):
        gate = F.silu(
            self.gate_projection(x)
        )
        value = self.value_projection(x)

        return self.output_projection(
            gate * value
        )


tokens = torch.randn(
    10,
    12,
    device=device,
)

dense_ffn = SwiGLUFFN(
    model_dim=12,
    hidden_dim=32,
).to(device)

dense_output = dense_ffn(tokens)

print("dense output:", dense_output.shape)


## 2. Generic top-k routed MoE

먼저 conventional MoE를 구현한다.

- router가 전체 expert에 대한 softmax score를 계산한다.
- top-k expert를 고른다.
- 선택된 expert만 token을 처리한다.
- 선택된 gate weight로 expert output을 합친다.


In [ ]:
class RoutedMoE(nn.Module):
    def __init__(
        self,
        model_dim,
        expert_hidden_dim,
        num_experts,
        top_k,
    ):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k

        self.router = nn.Linear(
            model_dim,
            num_experts,
            bias=False,
        )

        self.experts = nn.ModuleList(
            [
                SwiGLUFFN(
                    model_dim,
                    expert_hidden_dim,
                )
                for _ in range(num_experts)
            ]
        )

    def route(self, tokens):
        router_logits = self.router(tokens)

        router_probabilities = (
            router_logits.softmax(dim=-1)
        )

        topk_weights, topk_ids = (
            router_probabilities.topk(
                self.top_k,
                dim=-1,
            )
        )

        return (
            router_probabilities,
            topk_weights,
            topk_ids,
        )

    def forward(self, tokens):
        (
            router_probabilities,
            topk_weights,
            topk_ids,
        ) = self.route(tokens)

        output = torch.zeros_like(tokens)

        for slot in range(self.top_k):
            selected_ids = topk_ids[:, slot]
            selected_weights = (
                topk_weights[:, slot]
            )

            for expert_id, expert in enumerate(
                self.experts
            ):
                token_mask = (
                    selected_ids == expert_id
                )

                if not token_mask.any():
                    continue

                expert_output = expert(
                    tokens[token_mask]
                )

                output[token_mask] += (
                    selected_weights[
                        token_mask,
                        None,
                    ]
                    * expert_output
                )

        return (
            output,
            router_probabilities,
            topk_ids,
            topk_weights,
        )


generic_moe = RoutedMoE(
    model_dim=12,
    expert_hidden_dim=32,
    num_experts=4,
    top_k=1,
).to(device)

(
    generic_output,
    generic_probabilities,
    generic_ids,
    generic_weights,
) = generic_moe(tokens)

print("top-1 expert ids:", generic_ids.squeeze(-1))
print("output:", generic_output.shape)


## 3. Why fine-grained segmentation is not just 'more experts'

DeepSeekMoE starts from a coarse design with `N` experts and top-`K` activation.

Then each expert is split into `m` smaller experts.

- expert count: `N -> mN`
- activated experts: `K -> mK`
- each expert hidden width: approximately `d_ff -> d_ff / m`

따라서 expert 수는 늘지만 한 expert의 계산량은 줄어들어 activated FFN compute를 비슷하게 유지하면서 더 다양한 expert combination을 만들 수 있다.


In [ ]:
coarse_num_experts = 4
coarse_top_k = 2
coarse_hidden_dim = 32

segmentation_factor = 2

fine_num_experts = (
    segmentation_factor
    * coarse_num_experts
)
fine_top_k = (
    segmentation_factor
    * coarse_top_k
)
fine_hidden_dim = (
    coarse_hidden_dim
    // segmentation_factor
)

coarse_active_hidden = (
    coarse_top_k
    * coarse_hidden_dim
)
fine_active_hidden = (
    fine_top_k
    * fine_hidden_dim
)

print(
    "coarse:",
    coarse_num_experts,
    "experts, top-k =",
    coarse_top_k,
    "expert hidden =",
    coarse_hidden_dim,
)
print(
    "fine-grained:",
    fine_num_experts,
    "experts, top-k =",
    fine_top_k,
    "expert hidden =",
    fine_hidden_dim,
)
print(
    "active hidden work:",
    coarse_active_hidden,
    "vs",
    fine_active_hidden,
)


## 4. Paper-faithful tiny DeepSeekMoE

논문의 notation에 맞춰 전체 fine-grained expert 수를 `mN`이라 하자.

그 중 `K_s`개를 shared experts로 격리한다.

나머지 `mN - K_s`개만 routed pool에 들어가며 token마다 `mK - K_s`개를 선택한다.

최종 output은

`x + sum(shared experts) + sum(g_i * routed expert_i)`

이다.

원 논문 DeepSeekMoE는 routed pool 전체에 softmax를 적용한 score를 사용한다. 선택된 expert끼리 다시 별도의 softmax를 하는 Mixtral-style 구현으로 바꾸지 않는다.


In [ ]:
class TinyDeepSeekMoE(nn.Module):
    def __init__(
        self,
        model_dim=12,
        coarse_num_experts=4,
        coarse_top_k=2,
        segmentation_factor=2,
        shared_experts=1,
        coarse_hidden_dim=32,
    ):
        super().__init__()

        total_fine_experts = (
            segmentation_factor
            * coarse_num_experts
        )
        active_fine_experts = (
            segmentation_factor
            * coarse_top_k
        )

        routed_experts = (
            total_fine_experts
            - shared_experts
        )
        routed_top_k = (
            active_fine_experts
            - shared_experts
        )

        expert_hidden_dim = (
            coarse_hidden_dim
            // segmentation_factor
        )

        self.total_fine_experts = (
            total_fine_experts
        )
        self.shared_expert_count = (
            shared_experts
        )
        self.routed_expert_count = (
            routed_experts
        )
        self.routed_top_k = routed_top_k
        self.expert_hidden_dim = (
            expert_hidden_dim
        )

        self.shared_experts = nn.ModuleList(
            [
                SwiGLUFFN(
                    model_dim,
                    expert_hidden_dim,
                )
                for _ in range(
                    shared_experts
                )
            ]
        )

        self.routed_experts = nn.ModuleList(
            [
                SwiGLUFFN(
                    model_dim,
                    expert_hidden_dim,
                )
                for _ in range(
                    routed_experts
                )
            ]
        )

        self.router = nn.Linear(
            model_dim,
            routed_experts,
            bias=False,
        )

    def forward(self, tokens):
        shared_output = torch.zeros_like(
            tokens
        )

        for expert in self.shared_experts:
            shared_output = (
                shared_output
                + expert(tokens)
            )

        router_logits = self.router(tokens)

        routing_scores = (
            router_logits.softmax(dim=-1)
        )

        topk_weights, topk_ids = (
            routing_scores.topk(
                self.routed_top_k,
                dim=-1,
            )
        )

        routed_output = torch.zeros_like(
            tokens
        )

        for slot in range(
            self.routed_top_k
        ):
            selected_ids = topk_ids[
                :,
                slot,
            ]
            selected_weights = (
                topk_weights[
                    :,
                    slot,
                ]
            )

            for expert_id, expert in enumerate(
                self.routed_experts
            ):
                token_mask = (
                    selected_ids
                    == expert_id
                )

                if not token_mask.any():
                    continue

                expert_output = expert(
                    tokens[token_mask]
                )

                routed_output[token_mask] += (
                    selected_weights[
                        token_mask,
                        None,
                    ]
                    * expert_output
                )

        output = (
            tokens
            + shared_output
            + routed_output
        )

        return {
            "output": output,
            "shared_output": shared_output,
            "routed_output": routed_output,
            "routing_scores": routing_scores,
            "topk_ids": topk_ids,
            "topk_weights": topk_weights,
        }


deepseek_moe = TinyDeepSeekMoE().to(
    device
)

deepseek_result = deepseek_moe(tokens)

print(
    "total fine experts:",
    deepseek_moe.total_fine_experts,
)
print(
    "shared experts:",
    deepseek_moe.shared_expert_count,
)
print(
    "routed experts:",
    deepseek_moe.routed_expert_count,
)
print(
    "routed top-k:",
    deepseek_moe.routed_top_k,
)
print(
    "expert hidden dim:",
    deepseek_moe.expert_hidden_dim,
)
print(
    "output:",
    deepseek_result["output"].shape,
)


## 5. Routing load statistics and auxiliary balance term

원 논문은 expert collapse를 막기 위해 routed experts에 load-balance auxiliary loss를 둔다.

여기서는 작은 batch에서

- expert가 실제 선택된 비율
- router probability의 평균
- 두 값을 결합한 balance term

을 계산한다.


In [ ]:
topk_ids = deepseek_result["topk_ids"]
routing_scores = deepseek_result[
    "routing_scores"
]

selection_fraction = []

for expert_id in range(
    deepseek_moe.routed_expert_count
):
    selected = (
        topk_ids == expert_id
    ).float()

    selection_fraction.append(
        selected.mean()
    )

selection_fraction = torch.stack(
    selection_fraction
)

mean_router_probability = (
    routing_scores.mean(dim=0)
)

balance_term = (
    deepseek_moe.routed_expert_count
    * torch.sum(
        selection_fraction
        * mean_router_probability
    )
)

print(
    "selection fraction:",
    selection_fraction,
)
print(
    "mean router probability:",
    mean_router_probability,
)
print(
    "balance term:",
    balance_term.item(),
)


## 6. Gradient-flow sanity check

shared experts, router, 그리고 실제 선택된 routed experts까지 gradient가 연결되는지 확인한다.


In [ ]:
deepseek_moe.zero_grad(
    set_to_none=True
)

result = deepseek_moe(tokens)

loss = (
    result["output"].square().mean()
    + 0.01 * balance_term
)

loss.backward()

print(
    "router grad:",
    deepseek_moe.router.weight.grad.norm().item(),
)

print(
    "shared expert grad:",
    deepseek_moe.shared_experts[
        0
    ].gate_projection.weight.grad.norm().item(),
)

routed_grad_norms = []

for expert in deepseek_moe.routed_experts:
    gradient = (
        expert.gate_projection.weight.grad
    )

    routed_grad_norms.append(
        0.0
        if gradient is None
        else gradient.norm().item()
    )

print(
    "routed expert grad norms:",
    routed_grad_norms,
)


## References and provenance

**Switch / generic sparse MoE** — token-level routing, dispatch, sparse expert execution의 기본 구조를 보여준다.

**DeepSeekMoE** — fine-grained expert segmentation과 shared expert isolation을 모두 반영했다. `N -> mN`, `K -> mK`, expert FFN width 축소, `K_s` shared experts 격리, routed pool softmax와 top-k weighted sum을 작은 규모로 구현한다.

분산 expert parallelism, all-to-all communication, device-limited routing 같은 시스템 최적화는 T4 최소구현의 범위에서 제외하지만, expert topology와 routing 계산 그래프는 바꾸지 않는다.
